In [1]:

import os
import sys
import torch
import numpy as np

from pathlib import Path
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

PROJECT_DIR = Path(r"E:\Cotton-Disease-Project")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

from models.morph_cnn import MORPH_CNN

device = torch.device("cpu")

CHECKPOINT_PATH = (
    PROJECT_DIR / "checkpoints" / "best_model.pth"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

classes = checkpoint["classes"]

model = MORPH_CNN(
    num_classes=len(classes)
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

print("MORPH-CNN Prediction System Ready!")
print("Classes:", classes)


# Enter an image path from a separate,
# previously unseen cotton leaf image.

IMAGE_PATH = r"E:\Cotton-Disease-Project\sample_leaf.jpg"

def predict_disease(image_path):

    image = Image.open(image_path).convert("RGB")

    input_tensor = transform(image).unsqueeze(0)

    with torch.no_grad():

        output = model(input_tensor.to(device))

        probabilities = torch.softmax(
            output, dim=1
        )[0]

        predicted_index = probabilities.argmax().item()

    predicted_class = classes[predicted_index]

    confidence = probabilities[predicted_index].item() * 100

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(
        f"Prediction: {predicted_class}\n"
        f"Confidence: {confidence:.2f}%"
    )
    plt.show()

    print("Predicted Disease:", predicted_class)
    print(f"Confidence: {confidence:.2f}%")

    return predicted_class, confidence


# Run prediction
predict_disease(IMAGE_PATH)


MORPH-CNN Prediction System Ready!
Classes: ['Alternaria Leaf Spot', 'Bacterial Blight', 'Fusarium Wilt', 'Healthy Leaf', 'Verticillium Wilt']


FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Cotton-Disease-Project\\sample_leaf.jpg'